# Guided Lab: Build & Evaluate a RAG Assistant

This lab pairs with the **RAG Evaluation Metrics** session deck. You'll build a tiny
end-to-end RAG pipeline and implement each metric from the slides yourself, so you can
see exactly what Context Precision, Context Recall, Faithfulness, Answer Relevancy,
Hit Rate, and MRR are actually computing.

**Design choice for this lab:** we deliberately use only `numpy`, `pandas`, and
`scikit-learn`. No `faiss`, no `sentence-transformers`, no `ragas` — those pull in heavy
native binaries and model downloads that are the usual source of `OSError` /
segfault-style install issues in notebook environments. `scikit-learn`'s TF-IDF
vectorizer gives us a real retriever with zero download risk, so the whole lab runs
offline except for the optional LLM call in Part 4.

**What we'll build:**
1. A small knowledge base (same refund-policy example from the slides)
2. A TF-IDF retriever
3. An answer generator (OpenAI API if you have a key, otherwise a template fallback —
   the lab runs either way)
4. Every metric from the deck, implemented from scratch
5. An evaluation harness that scores a batch of test queries into one summary table

> **Next session:** we'll wrap this exact pipeline in a **Gradio** UI so it's
> click-and-type instead of run-the-cell. Nothing here changes when we do that — Gradio
> just becomes a thin interface on top of the functions you write today.


## Part 0 — Setup

Run this once. If you're in a fresh environment, uncomment the pip line.


In [1]:
# !pip install -q numpy pandas scikit-learn

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 100)
print("Ready.")


Ready.


## Part 1 — Knowledge base

A handful of short documents about a fictional product's refund policy — the same
scenario used in the slide examples, so the metric scores you get here will feel
familiar.


In [2]:
documents = [
    {"id": "d1", "text": "Annual plan refund policy: customers may request a full refund within 30 days of purchasing an annual subscription."},
    {"id": "d2", "text": "Monthly plan pricing tiers: Basic is $9/month, Pro is $29/month, and Team is $49/month per seat."},
    {"id": "d3", "text": "How to cancel a subscription: go to Account Settings > Billing > Cancel Plan. Cancellation takes effect at the end of the billing period."},
    {"id": "d4", "text": "Refund exceptions for enterprise annual contracts: enterprise annual plans have a 45-day refund window, but require written manager approval before processing."},
    {"id": "d5", "text": "Data export: users can export all their data as a CSV or JSON file at any time from Account Settings > Data Export."},
    {"id": "d6", "text": "Support hours: live chat support is available Monday-Friday, 9am-6pm ET. Email support is available 24/7 with a 24-hour response target."},
]

df_docs = pd.DataFrame(documents)
df_docs


,id,text
0,d1,Annual plan refund policy: customers may request a full refund within 30 days of purchasing an a...
1,d2,"Monthly plan pricing tiers: Basic is $9/month, Pro is $29/month, and Team is $49/month per seat."
2,d3,How to cancel a subscription: go to Account Settings > Billing > Cancel Plan. Cancellation takes...
3,d4,Refund exceptions for enterprise annual contracts: enterprise annual plans have a 45-day refund ...
4,d5,Data export: users can export all their data as a CSV or JSON file at any time from Account Sett...
5,d6,"Support hours: live chat support is available Monday-Friday, 9am-6pm ET. Email support is availa..."


## Part 2 — A simple retriever (TF-IDF + cosine similarity)

This is the "Retriever" box from the pipeline diagram in the deck. TF-IDF turns each
document into a sparse vector of word-importance scores; cosine similarity ranks
documents against a query vector built the same way.


In [3]:
vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(df_docs["text"])

def retrieve(query, k=3):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, doc_matrix)[0]
    ranked_idx = np.argsort(scores)[::-1][:k]
    results = df_docs.iloc[ranked_idx].copy()
    results["score"] = scores[ranked_idx]
    return results.reset_index(drop=True)

# quick smoke test
retrieve("What is the refund window for annual plans?", k=4)


,id,text,score
0,d4,Refund exceptions for enterprise annual contracts: enterprise annual plans have a 45-day refund ...,0.568016
1,d1,Annual plan refund policy: customers may request a full refund within 30 days of purchasing an a...,0.415451
2,d5,Data export: users can export all their data as a CSV or JSON file at any time from Account Sett...,0.000000
3,d6,"Support hours: live chat support is available Monday-Friday, 9am-6pm ET. Email support is availa...",0.000000


### 💬 Discussion checkpoint 1

Look at the scores above.

- Which documents are genuinely relevant to the refund question, and which just share
  vocabulary (e.g. "plan", "annual")?
- If you set `k=3`, how many of those 4 are actually relevant? That fraction *is*
  Context Precision — you'll compute it properly in Part 5.


## Part 3 — Generate an answer

This plugs into the "LLM" box in the pipeline. If you have an `OPENAI_API_KEY` set as
an environment variable, this will call the real API. If not, it falls back to a
simple extractive template so the rest of the lab still runs end-to-end — the
generation quality will be lower, but every metric below still computes normally.


In [4]:
import os

def _template_fallback_answer(query, context_chunks):
    # Naive extractive fallback: stitch together the most relevant sentences.
    combined = " ".join(context_chunks)
    return f"Based on the available information: {combined}"

def generate_answer(query, context_chunks, model="gpt-4o-mini"):
    api_key = os.environ.get("OPENAI_API_KEY")
    if not api_key:
        return _template_fallback_answer(query, context_chunks)

    try:
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        context_text = "\n".join(f"- {c}" for c in context_chunks)
        prompt = (
            f"Answer the question using ONLY the context below. "
            f"If the context doesn't contain the answer, say so.\n\n"
            f"Context:\n{context_text}\n\nQuestion: {query}\nAnswer:"
        )
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"[warning] LLM call failed ({e}); using template fallback.")
        return _template_fallback_answer(query, context_chunks)

# try it
query = "What is the refund window for annual plans?"
top_chunks = retrieve(query, k=4)["text"].tolist()
answer = generate_answer(query, top_chunks)
print(answer)


Based on the available information: Refund exceptions for enterprise annual contracts: enterprise annual plans have a 45-day refund window, but require written manager approval before processing. Annual plan refund policy: customers may request a full refund within 30 days of purchasing an annual subscription. Data export: users can export all their data as a CSV or JSON file at any time from Account Settings > Data Export. Support hours: live chat support is available Monday-Friday, 9am-6pm ET. Email support is available 24/7 with a 24-hour response target.


## Part 4 — Labeled eval set

To score Context Recall and Answer Correctness you need ground truth: which document
IDs are actually relevant to each query, and what the reference answer is. This is the
same idea as the "Ground truth" line in the Context Recall slide example.


In [5]:
eval_set = [
    {
        "query": "What is the refund window for annual plans?",
        "relevant_doc_ids": ["d1", "d4"],
        "reference_answer": "Annual plans have a 30-day refund window. Enterprise annual plans have a 45-day window but require manager approval.",
    },
    {
        "query": "How do I cancel my subscription?",
        "relevant_doc_ids": ["d3"],
        "reference_answer": "Go to Account Settings > Billing > Cancel Plan. The cancellation takes effect at the end of the current billing period.",
    },
    {
        "query": "What are your support hours?",
        "relevant_doc_ids": ["d6"],
        "reference_answer": "Live chat is available Monday-Friday 9am-6pm ET, and email support is available 24/7 with a 24-hour response target.",
    },
]

pd.DataFrame(eval_set)


,query,relevant_doc_ids,reference_answer
0,What is the refund window for annual plans?,"[d1, d4]",Annual plans have a 30-day refund window. Enterprise annual plans have a 45-day window but requi...
1,How do I cancel my subscription?,[d3],Go to Account Settings > Billing > Cancel Plan. The cancellation takes effect at the end of the ...
2,What are your support hours?,[d6],"Live chat is available Monday-Friday 9am-6pm ET, and email support is available 24/7 with a 24-h..."


## Part 5 — Retrieval metrics: Context Precision & Context Recall

Direct implementations of the formulas from the slides:

```
Precision@K = relevant chunks retrieved / total chunks retrieved
Recall      = relevant docs retrieved   / total relevant docs that exist
```


In [6]:
def context_precision(retrieved_ids, relevant_ids):
    if not retrieved_ids:
        return 0.0
    hits = sum(1 for rid in retrieved_ids if rid in relevant_ids)
    return hits / len(retrieved_ids)

def context_recall(retrieved_ids, relevant_ids):
    if not relevant_ids:
        return None  # undefined, not zero
    hits = sum(1 for rid in relevant_ids if rid in retrieved_ids)
    return hits / len(relevant_ids)

# demo on the first eval example
k = 4
example = eval_set[0]
retrieved = retrieve(example["query"], k=k)
retrieved_ids = retrieved["id"].tolist()

print("Retrieved:", retrieved_ids)
print("Relevant :", example["relevant_doc_ids"])
print(f"Context Precision@{k}:", round(context_precision(retrieved_ids, example["relevant_doc_ids"]), 2))
print("Context Recall:", round(context_recall(retrieved_ids, example["relevant_doc_ids"]), 2))


Retrieved: ['d4', 'd1', 'd5', 'd6']
Relevant : ['d1', 'd4']
Context Precision@4: 0.5
Context Recall: 1.0


## Part 6 — Ranking metrics: Hit Rate & MRR

From the "Bonus: Ranking Metrics" slide — these care about *where* the relevant chunk
lands, not just whether it's in the top K at all.


In [7]:
def hit_rate(retrieved_ids, relevant_ids):
    return 1.0 if any(rid in relevant_ids for rid in retrieved_ids) else 0.0

def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, rid in enumerate(retrieved_ids, start=1):
        if rid in relevant_ids:
            return 1.0 / rank
    return 0.0

print("Hit Rate:", hit_rate(retrieved_ids, example["relevant_doc_ids"]))
print("Reciprocal Rank:", round(reciprocal_rank(retrieved_ids, example["relevant_doc_ids"]), 2))


Hit Rate: 1.0
Reciprocal Rank: 1.0


### 💬 Discussion checkpoint 2

- Try `k=2` instead of `k=4` above (edit and re-run Part 5). What happens to Precision?
  What happens to Recall?
- Can you construct a query where Hit Rate = 1.0 but Reciprocal Rank is low? What does
  that combination tell you about your ranking, versus your retrieval?


## Part 7 — Generation metrics: Faithfulness & Answer Relevancy

We don't have a labeled "claims" dataset, so we use lightweight TF-IDF cosine-similarity
proxies — same tool we already built, applied differently:

- **Faithfulness proxy**: how similar is the *answer* to the *retrieved context*? A
  low score is a hallucination warning sign — the answer is talking about things the
  context never mentioned.
- **Answer Relevancy proxy**: how similar is the *answer* to the *query*? A low score
  means the answer may be true but never actually addresses the question.

These are simplified stand-ins for what libraries like **RAGAS** or **TruLens** do with
LLM-graded claim extraction — worth adopting once you outgrow this lab, but the
intuition you build here (grounding vs. relevance are *different axes*) transfers
directly.


In [8]:
def _tfidf_cosine(text_a, text_b):
    # Build a small local vectorizer over just these two texts so we're not
    # dependent on vocabulary from the document corpus.
    local_vec = TfidfVectorizer(stop_words="english").fit([text_a, text_b])
    vecs = local_vec.transform([text_a, text_b])
    return float(cosine_similarity(vecs[0], vecs[1])[0][0])

def faithfulness_proxy(answer, context_chunks):
    context_text = " ".join(context_chunks)
    return _tfidf_cosine(answer, context_text)

def answer_relevancy_proxy(answer, query):
    return _tfidf_cosine(answer, query)

print("Faithfulness proxy:", round(faithfulness_proxy(answer, top_chunks), 2))
print("Answer Relevancy proxy:", round(answer_relevancy_proxy(answer, query), 2))


Faithfulness proxy: 0.98
Answer Relevancy proxy: 0.36


### 💬 Discussion checkpoint 3

- Manually edit `answer` to a made-up sentence that has nothing to do with the context
  (e.g. `"Our office is located in Berlin."`) and re-run the Faithfulness proxy. Does the
  score drop the way you'd expect?
- Now edit it to a true-but-off-topic sentence (e.g. quoting `d6`'s support hours while
  the query is about refunds). Which proxy catches that, Faithfulness or Relevancy?


## Part 8 — Put it all together: evaluation harness

Run every metric over the full eval set and summarize into one table — this mirrors
the cheat-sheet slide from the deck, but with your own numbers.


In [9]:
def evaluate_pipeline(eval_set, k=4):
    rows = []
    for ex in eval_set:
        retrieved = retrieve(ex["query"], k=k)
        retrieved_ids = retrieved["id"].tolist()
        context_chunks = retrieved["text"].tolist()
        ans = generate_answer(ex["query"], context_chunks)

        rows.append({
            "query": ex["query"],
            "context_precision": round(context_precision(retrieved_ids, ex["relevant_doc_ids"]), 2),
            "context_recall": round(context_recall(retrieved_ids, ex["relevant_doc_ids"]), 2),
            "hit_rate": hit_rate(retrieved_ids, ex["relevant_doc_ids"]),
            "reciprocal_rank": round(reciprocal_rank(retrieved_ids, ex["relevant_doc_ids"]), 2),
            "faithfulness_proxy": round(faithfulness_proxy(ans, context_chunks), 2),
            "answer_relevancy_proxy": round(answer_relevancy_proxy(ans, ex["query"]), 2),
            "answer_correctness_proxy": round(_tfidf_cosine(ans, ex["reference_answer"]), 2),
        })
    return pd.DataFrame(rows)

results = evaluate_pipeline(eval_set, k=4)
results


,query,context_precision,context_recall,hit_rate,reciprocal_rank,faithfulness_proxy,answer_relevancy_proxy,answer_correctness_proxy
0,What is the refund window for annual plans?,0.50,1.0,1.0,1.0,0.98,0.36,0.38
1,How do I cancel my subscription?,0.25,1.0,1.0,1.0,0.98,0.20,0.33
2,What are your support hours?,0.25,1.0,1.0,1.0,0.98,0.20,0.41


In [10]:
# Summary row — the average of each column, like a dashboard tile per metric
summary = results.drop(columns=["query"]).mean(numeric_only=True).round(2)
summary.to_frame("average").rename_axis("metric")


,average
metric,
context_precision,0.33
context_recall,1.00
hit_rate,1.00
reciprocal_rank,1.00
faithfulness_proxy,0.98
answer_relevancy_proxy,0.25
answer_correctness_proxy,0.37


## Part 9 — Try it yourself

- Add 2-3 more documents and eval queries of your own.
- Change `k` in `evaluate_pipeline` and watch Precision and Recall move in opposite
  directions — this is the trade-off from Discussion checkpoint 1 in the slides.
- If you have an `OPENAI_API_KEY`, set it and re-run Part 8 — compare the real LLM's
  Faithfulness/Relevancy proxies against the template fallback's.

## What's next

Next session, we'll take `retrieve()`, `generate_answer()`, and `evaluate_pipeline()`
exactly as they are and wrap them in a **Gradio** app: a text box for the query, a
panel showing retrieved chunks, the generated answer, and a live metrics table. No
changes to the logic above — Gradio just becomes the front door.
